# P1_01 · Data check

Runs `scripts/check_data.py` for Kvasir-SEG and COCO val2017 on Colab and compares the result with the versions committed to the repo. Expected: Kvasir 1000 image/mask pairs, `CHECK OK`, split files and `docs/data_stats.md` unchanged.

Needs P1_00 first.

**Before you start** (once per Google account):
1. Colab left bar → 🔑 *Secrets* → add `GH_TOKEN` (a GitHub fine-grained token with read access
   to `geekRoy0126/promptshift`) and switch on *Notebook access*. Never paste the token into a cell.
2. *Runtime → Change runtime type → T4 GPU*.
3. *Runtime → Run all*. Allow Google Drive access when asked.

All data, weights and results go to `MyDrive/promptshift/`, so a disconnect loses nothing:
rerun the notebook and finished steps are skipped.

In [ ]:
# Parameters (edit before Run all if needed)
DATASETS = ["kvasir_seg", "coco_val2017"]

In [ ]:
# SETUP: identical in every notebook (generated by scripts/build_notebooks.py).
import base64
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/geekRoy0126/promptshift.git"
REF = "main"  # branch, tag or commit to run

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    REPO = Path("/content/promptshift")
    try:
        _token = userdata.get("GH_TOKEN")
    except Exception as e:  # missing secret or notebook access not granted
        raise RuntimeError(
            "Add the Colab secret GH_TOKEN (key icon in the left bar) and switch on "
            "'Notebook access' for it."
        ) from e
    _auth = base64.b64encode(f"x-access-token:{_token}".encode()).decode()

    def _git(*args, auth=False):
        pre = ["-c", f"http.extraHeader=Authorization: Basic {_auth}"] if auth else []
        r = subprocess.run(["git", *pre, "-C", str(REPO), *args], capture_output=True, text=True)
        if r.returncode:
            msg = (r.stderr or r.stdout).replace(_token, "***").replace(_auth, "***")
            raise RuntimeError(f"git {args[0]} failed: {msg}")

    REPO.mkdir(parents=True, exist_ok=True)
    if not (REPO / ".git").is_dir():
        _git("init", "-q")
    _git("fetch", "-q", "--depth", "1", REPO_URL, REF, auth=True)
    _git("reset", "-q", "--hard", "FETCH_HEAD")  # throwaway clone; all outputs live on Drive
    del _token, _auth
    _pip = [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)]
    r = subprocess.run(_pip, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError("pip install -e failed (log above)")
else:  # local CPU run: use the checkout this notebook lives in
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
os.environ["PYTHONUNBUFFERED"] = "1"
# scripts must import this checkout's code, whatever else is installed
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO / "src"), os.environ.get("PYTHONPATH", "")])


def run(*args, check=True):
    """Run a repo script with live output; stop the notebook if it fails. Returns the output."""
    cmd = [sys.executable, *map(str, args)]
    print("$ python", " ".join(cmd[1:]), flush=True)
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = []
    while chunk := p.stdout.read1(4096):
        text = chunk.decode(errors="replace")
        out.append(text)
        sys.stdout.write(text)
        sys.stdout.flush()
    if p.wait() and check:
        raise RuntimeError(f"exit code {p.returncode}: python {' '.join(cmd[1:])}")
    return "".join(out)


def check_gpu():
    import torch

    if IN_COLAB and not torch.cuda.is_available():
        raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then Run all again.")
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")


def git_out(*args):
    r = subprocess.run(["git", "-C", str(REPO), *args], capture_output=True, text=True)
    return r.stdout.strip()


SHA = git_out("rev-parse", "--short", "HEAD")
print(f"repo {REPO} @ {SHA} (Colab: {IN_COLAB})")

## 1. Check each dataset

In [ ]:
from IPython.display import Image, display

for name in DATASETS:
    out = run("scripts/check_data.py", "--dataset", name)
    assert "CHECK OK" in out, f"{name}: data check failed (see output above)"
    display(Image(filename=str(REPO / "figures" / f"data_{name}_samples.png")))

## 2. Compare with the committed versions
The split files must be identical. Any change in `docs/data_stats.md` is printed and saved to `MyDrive/promptshift/results/colab_check/`; then the clone is reset so the next notebooks run on a clean tree (their `git_sha` would otherwise end in `-dirty`).

In [ ]:
import shutil

from promptshift.paths import get_paths

check_dir = get_paths().results / "colab_check"
check_dir.mkdir(parents=True, exist_ok=True)
split_diff = git_out("diff", "--stat", "--", "splits")
stats_diff = git_out("diff", "--", "docs/data_stats.md")
(check_dir / "data_stats.diff").write_text(stats_diff)
for name in DATASETS:
    fig = REPO / "figures" / f"data_{name}_samples.png"
    shutil.copy(fig, check_dir / fig.name)
print("docs/data_stats.md:", "unchanged" if not stats_diff else "CHANGED")
print(stats_diff)
subprocess.run(["git", "-C", str(REPO), "checkout", "--", "docs", "figures"],
               check=True)
assert not split_diff, f"split files changed on Colab:\n{split_diff}"
print("\nP1_01 done. Next: P1_02_medsam_parity.")